In [4]:
import duckdb

In [5]:
con = duckdb.connect(database='dados_duckdb.db', read_only=False)

In [6]:
df = con.execute("""
                 Select *
                 from(
                 select *, Row_number() over (Partition by NATBR order by data_ingestão DESC) as row
                from bronze_z0019
                 where data_ingestão >= '2026-02-17'
                 ) Where row = 1
                 """).fetchdf()
df.head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,NATBR,MAKTX,WERKS,MAINS,LABST,nome_arquivo,data_ingestão,row
0,1005,MACHADO,BT50,100,100,z0019_2.csv,2026-02-17 08:09:59.307653,1
1,1001,PARAFUSO,BT10,100,100,z0019_1.csv,2026-02-17 07:01:04.013625,1
2,1003,PREGO,BT10,100,60,z0019_2.csv,2026-02-17 08:09:59.307653,1
3,1002,MARTELO,BT50,100,1500,z0019_1.csv,2026-02-17 07:01:04.013625,1
4,1004,SERRA,BT50,100,200,z0019_2.csv,2026-02-17 08:09:59.307653,1


In [32]:
df_final = df.drop(columns=['nome_arquivo', 'data_ingestão', 'row'])
df_final = df_final.rename(columns={"NATBR":"id"})
df_final = df_final.rename(columns={"MAKTX":"nm_produto"})
df_final = df_final.rename(columns={"WERKS":"id_categoria"})
df_final = df_final.rename(columns={"MAINS":"id_fornecedor"})
df_final = df_final.rename(columns={"LABST":"vl_preço"})
df_final.head(10)

,id,nm_produto,id_categoria,id_fornecedor,vl_preço
0,1005,MACHADO,BT50,100,100
1,1001,PARAFUSO,BT10,100,100
2,1003,PREGO,BT10,100,60
3,1002,MARTELO,BT50,100,1500
4,1004,SERRA,BT50,100,200


In [33]:
df_final.dtypes

id               object
nm_produto       object
id_categoria     object
id_fornecedor    object
vl_preço         object
dtype: object

In [35]:
df2 = df_final
df2 = df2.astype(
    {
        'id': int,
        'nm_produto': str,
        'id_categoria': str,
        'id_fornecedor':int,
        'vl_preço': float
    }
)

df2.dtypes
# df2.head(10)

id                 int64
nm_produto        object
id_categoria      object
id_fornecedor      int64
vl_preço         float64
dtype: object

In [36]:
con.execute("""
Create table if not exists produtos(
            id bigint,
            nm_produto text,
            id_categoria text,
            id_fornecedor bigint,
            vl_preço float
            )
""")

In [37]:
df2.head(10)

,id,nm_produto,id_categoria,id_fornecedor,vl_preço
0,1005,MACHADO,BT50,100,100.0
1,1001,PARAFUSO,BT10,100,100.0
2,1003,PREGO,BT10,100,60.0
3,1002,MARTELO,BT50,100,1500.0
4,1004,SERRA,BT50,100,200.0


In [40]:
con.execute("INSERT INTO produtos SELECT * FROM df2")

In [41]:
df_resultado = con.execute ("select * from produtos").fetchdf()
df_resultado.head(10)

,id,nm_produto,id_categoria,id_fornecedor,vl_preço
0,1005,MACHADO,BT50,100,100.0
1,1001,PARAFUSO,BT10,100,100.0
2,1003,PREGO,BT10,100,60.0
3,1002,MARTELO,BT50,100,1500.0
4,1004,SERRA,BT50,100,200.0


In [42]:
con.close()